In [1]:
import pandas as pd

orders = pd.read_csv("../data/raw/ecommerce_data/olist_orders_dataset.csv")
customers = pd.read_csv("../data/raw/ecommerce_data/olist_customers_dataset.csv")
order_items = pd.read_csv("../data/raw/ecommerce_data/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/ecommerce_data/olist_order_payments_dataset.csv")
leads = pd.read_csv("../data/raw/marketing_funnel_data/olist_marketing_qualified_leads_dataset.csv")
deals = pd.read_csv("../data/raw/marketing_funnel_data/olist_closed_deals_dataset.csv")

for name, df in [("orders", orders), ("customers", customers), ("order_items", order_items),
                  ("payments", payments), ("leads", leads), ("deals", deals)]:
    print(f"\n{name}: {df.shape}")
    print(df.columns.tolist())


orders: (99441, 8)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

customers: (99441, 5)
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

order_items: (112650, 7)
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

payments: (103886, 5)
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

leads: (8000, 4)
['mql_id', 'first_contact_date', 'landing_page_id', 'origin']

deals: (842, 14)
['mql_id', 'seller_id', 'sdr_id', 'sr_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile', 'has_company', 'has_gtin', 'average_stock', 'business_type', 'declared_product_catalog_size', 'declared_monthly_revenue']


In [2]:
import sys
sys.path.append('..')

In [3]:
import numpy as np

In [4]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
leads.head()

,mql_id,first_contact_date,landing_page_id,origin
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search


In [6]:
leads['origin'].nunique()


10

In [7]:
leads['origin'].value_counts()


origin
organic_search       2296
paid_search          1586
social               1350
unknown              1099
direct_traffic        499
email                 493
referral              284
other                 150
display               118
other_publicities      65
Name: count, dtype: int64

In [8]:
deals.head()

,mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
0,5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0
1,a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,09285259593c61296eef10c734121d5b,d3d1e91a157ea7f90548eef82f1955e3,2018-05-08 20:17:59,car_accessories,industry,eagle,NaN,NaN,NaN,reseller,NaN,0.0
2,327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,b90f87164b5f8c2cfa5c8572834dbe3f,6565aa9ce3178a5caf6171827af3a9ba,2018-06-05 17:27:23,home_appliances,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.0
3,f5fee8f7da74f4887f5bcae2bafb6dd6,21e1781e36faf92725dde4730a88ca0f,56bf83c4bb35763a51c2baab501b4c67,d3d1e91a157ea7f90548eef82f1955e3,2018-01-17 13:51:03,food_drink,online_small,NaN,NaN,NaN,NaN,reseller,NaN,0.0
4,ffe640179b554e295c167a2f6be528e0,ed8cb7b190ceb6067227478e48cf8dde,4b339f9567d060bcea4f5136b9f5949e,d3d1e91a157ea7f90548eef82f1955e3,2018-07-03 20:17:45,home_appliances,industry,wolf,NaN,NaN,NaN,manufacturer,NaN,0.0


In [9]:
import src.db as db


In [10]:
engine = db.get_engine()

DATABASE_URL is set.


In [11]:
query = """
select seller_id, origin, sum(price) as total_revenue
from seller_channel_revenue
group by seller_id, origin

"""

In [12]:
seller_revenue_df = pd.read_sql(query, engine)
print(seller_revenue_df.shape)
seller_revenue_df.head()

(380, 3)


,seller_id,origin,total_revenue
0,596849622429351f47b32e6cae1055ff,unknown,1104.70
1,e1643dff33666ca629e3644c02738179,paid_search,170.00
2,0ddefe3c7a032b91f4e25b9c3a08fca1,paid_search,402.90
3,28872dc528e978a639754bc8c2ce5a4c,paid_search,1009.90
4,382229d1e840115ffe3dbf5ff460e417,organic_search,4096.96


In [13]:
seller_revenue_df.head()

,seller_id,origin,total_revenue
0,596849622429351f47b32e6cae1055ff,unknown,1104.70
1,e1643dff33666ca629e3644c02738179,paid_search,170.00
2,0ddefe3c7a032b91f4e25b9c3a08fca1,paid_search,402.90
3,28872dc528e978a639754bc8c2ce5a4c,paid_search,1009.90
4,382229d1e840115ffe3dbf5ff460e417,organic_search,4096.96


In [14]:
seller_revenue_df = seller_revenue_df.rename(columns={'sum': 'total_revenue'})

In [15]:
channel_stats = seller_revenue_df.groupby(['origin']).agg(
    average_revenue=('total_revenue', 'mean'),
    seller_count=('seller_id', 'count'),
    std_dev=('total_revenue', 'std')
)

channel_stats['std_error'] = channel_stats['std_dev']/np.sqrt(channel_stats['seller_count'])

In [16]:
channel_stats

,average_revenue,seller_count,std_dev,std_error
origin,,,,
direct_traffic,706.577419,31,1152.483861,206.992212
display,461.500000,2,475.882864,336.500000
email,1414.165000,6,2652.907939,1083.045131
organic_search,1832.065929,113,4937.133865,464.446486
other,3444.325000,2,3335.458042,2358.525000
paid_search,1537.396535,101,3903.474048,388.410185
referral,1987.461111,9,2317.927872,772.642624
social,1402.515806,31,2619.887002,470.545596
unknown,2529.238824,85,12361.832895,1340.829035


In [17]:
from scipy import stats


In [19]:

def get_confidence_interval(row):
    lower, upper = stats.t.interval(
        confidence=0.95,
        df=row['seller_count'] - 1,
        loc=row['average_revenue'],
        scale=row['std_error']
    )
    return pd.Series({'ci_lower': lower, 'ci_upper': upper})

channel_stats[['ci_lower', 'ci_upper']] = channel_stats.apply(get_confidence_interval, axis=1)

In [20]:
channel_stats

,average_revenue,seller_count,std_dev,std_error,ci_lower,ci_upper
origin,,,,,,
direct_traffic,706.577419,31,1152.483861,206.992212,283.842926,1129.311913
display,461.500000,2,475.882864,336.500000,-3814.137894,4737.137894
email,1414.165000,6,2652.907939,1083.045131,-1369.891140,4198.221140
organic_search,1832.065929,113,4937.133865,464.446486,911.824753,2752.307105
other,3444.325000,2,3335.458042,2358.525000,-26523.576525,33412.226525
paid_search,1537.396535,101,3903.474048,388.410185,766.801790,2307.991279
referral,1987.461111,9,2317.927872,772.642624,205.744025,3769.178197
social,1402.515806,31,2619.887002,470.545596,441.533496,2363.498117
unknown,2529.238824,85,12361.832895,1340.829035,-137.146757,5195.624404
